# 水印 Mask 检测：可视化

四栏：**水印图 / 原图 / 水印 mask / overlay**。

推理走同目录的 `infer.py`（门控融合 + 尺度匹配，无后处理）。
数据目录在第一个代码块顶部指定，里面放 `source/` 和 `candidate/` 两个子目录即可。


In [ ]:
%matplotlib inline
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
# 中文字体，否则标题里的中文会渲染成方块
matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK JP', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
import torch
from PIL import Image

# ---- 只改这一处：数据目录 --------------------------------------------------
# 目录里按 <组名>/source 和 <组名>/candidate 放图，按文件名主干配对。
# 设为 None 时默认读取 handoff/test_data_pairs。
DATA_DIR = None
# ---------------------------------------------------------------------------

THRESHOLD = 0.35   # 概率阈值，偏召回（mask 外会回退到原图，漏检比过检严重）
MAX_IMAGES = 0     # 0 = 全部

# 定位本模型目录（代码不依赖目录叫什么名字）
MODEL_DIR = Path.cwd().resolve()
if not (MODEL_DIR / 'infer.py').is_file():
    for candidate in ('mask-model', 'handoff', 'mask-model/handoff', '.'):
        path = Path(candidate)
        if (path / 'infer.py').is_file():
            MODEL_DIR = path.resolve()
            break
assert (MODEL_DIR / 'infer.py').is_file(), f'找不到 infer.py（当前目录 {Path.cwd()}）'
if DATA_DIR is None:
    DATA_DIR = MODEL_DIR / 'test_data_pairs'
sys.path.insert(0, str(MODEL_DIR))
# 强制重新加载 infer / models：Jupyter 内核会缓存已导入的模块，
# 改了 infer.py 之后不重启内核会继续用旧版本。
for _name in [n for n in list(sys.modules)
              if n in ('infer', 'models') or n.startswith('models.')]:
    sys.modules.pop(_name, None)
from infer import load_model, predict, TILE, OVERLAP

model, DEVICE, INFO = load_model(MODEL_DIR / 'weights/watermark_mask.pt')
print('model     :', MODEL_DIR)
print('data      :', DATA_DIR)
print('architecture=%s device=%s' % (INFO['architecture'], DEVICE))


In [ ]:
# 按类别子目录扫描，文件名主干配对 source/ 与 candidate/
assert DATA_DIR.is_dir(), f'数据目录不存在: {DATA_DIR}'
pairs = []
for category_dir in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
    src_dir = category_dir / 'source'
    cand_dir = category_dir / 'candidate'
    if not src_dir.is_dir():
        continue
    for source_path in sorted(src_dir.iterdir()):
        candidate_path = cand_dir / f'{source_path.stem}.png'
        if candidate_path.is_file():
            pairs.append((source_path, candidate_path))
if MAX_IMAGES:
    pairs = pairs[:MAX_IMAGES]
print(f'{len(pairs)} 对图像')

results = []
for index, (source_path, candidate_path) in enumerate(pairs, 1):
    with Image.open(source_path) as handle:
        source = handle.convert('RGB')
    with Image.open(candidate_path) as handle:
        candidate = handle.convert('RGB')
    out = predict(model, source, candidate, DEVICE, tile=TILE, overlap=OVERLAP)
    probability = out['probability']
    mask = probability >= THRESHOLD

    overlay = np.asarray(source, dtype=np.float32) / 255.0
    overlay[mask] = 0.3 * overlay[mask] + 0.7 * np.array([0.0, 0.85, 1.0])

    results.append({
        'id': source_path.stem,
        'source': np.asarray(source),
        'candidate': np.asarray(candidate),
        'mask': mask,
        'overlay': (overlay.clip(0, 1) * 255).astype(np.uint8),
        'mask_fraction': float(mask.mean()),
        'scale_mismatch': out['scale_mismatch'],
        'matched_scale': out['matched_scale'],
    })
    print(f"[{index}/{len(pairs)}] {source_path.stem}: mask={mask.mean()*100:.2f}% "
          f"mismatch={out['scale_mismatch']:.2f}x matched={out['matched_scale']}")
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


In [ ]:
def show_results(limit=9):
    """四栏：水印图 / 原图 / 水印 mask / overlay。"""
    selected = results[:limit] if limit else results
    if not selected:
        print('没有可显示的结果')
        return
    fig, axes = plt.subplots(len(selected), 4, figsize=(20, 4.8 * len(selected)))
    axes = np.atleast_2d(axes)
    for row, result in enumerate(selected):
        panels = (
            (result['source'], 'watermarked source'),
            (result['candidate'], 'clean candidate'),
            (result['mask'], f"watermark mask ({result['mask_fraction'] * 100:.2f}%)"),
            (result['overlay'], 'overlay'),
        )
        for col, (image, title) in enumerate(panels):
            axes[row, col].imshow(image, cmap='gray' if col == 2 else None,
                                  vmin=0 if col == 2 else None, vmax=1 if col == 2 else None)
            axes[row, col].set_title(f"{title}\n{result['id']}", fontsize=9)
            axes[row, col].axis('off')
    plt.tight_layout()
    plt.show()

show_results(limit=9)
display(pd.DataFrame([{k: v for k, v in r.items()
                       if k in ('id', 'mask_fraction', 'scale_mismatch', 'matched_scale')}
                      for r in results]))
